# Week 2 · Module 4 Practical — Build the Judge ⚖️

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 2 · Module 4: Evaluation deep dive — retrieval metrics & RAGAS-style LLM-as-judge**

---

### What you'll do in the next 30 minutes

| # | Experiment | Metric built |
|---|-----------|--------------|
| 1 | Rebuild v0.3 (fast) | — (condensed setup) |
| 2 | Catch a hallucination red-handed | **Faithfulness** (claim decomposition) |
| 3 | The reverse-question trick | **Answer relevancy** (Day 2's embeddings!) |
| 4 | Grade the context | **Context precision** & **context recall** |
| 5 | The full report card | **EvalHarness** → tomorrow's deliverable table |

> ⚖️ We build RAGAS-style metrics **by hand** so you own the mechanics. The `ragas` pip library packages the same ideas — try it after class; the concepts transfer exactly.
> 💰 Judge calls use the LLM more than usual labs — still well under $1 for the whole notebook.

## Step 0 — Setup + rebuild RAGBot v0.3 (condensed) 🔑

Everything from Days 1–3 in two cells, so today's focus stays on evaluation.

In [ ]:
%pip install -q -U openai rank_bm25 chromadb

In [ ]:
from getpass import getpass
from openai import OpenAI
import numpy as np, chromadb, json
from rank_bm25 import BM25Okapi

API_KEY = getpass("Paste the OpenAI API key: ")
client = OpenAI(api_key=API_KEY)
CHAT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

def ask(prompt, temperature=0.0, max_tokens=400):
    r = client.chat.completions.create(model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content.strip()

def embed_batch(texts):
    r = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in r.data])

# --- the knowledge base (Days 1-3) ---
KB = [
    ("P-101", "Prima Electric Kettle 1.5L | Rs. 1,299 | 1500W fast boil, auto shut-off, 1-year warranty | in stock: 42", "kitchen", 1299),
    ("P-102", "Prima Electric Kettle 2.0L | Rs. 1,699 | family size, keep-warm mode, 1-year warranty | in stock: 18", "kitchen", 1699),
    ("P-103", "Zenith Steel Kettle 1.0L | Rs. 899 | budget stovetop kettle, no electricity needed | in stock: 65", "kitchen", 899),
    ("P-104", "Zenith Pro Kettle 1.8L | Rs. 2,499 | premium, temperature control, 2-year warranty | in stock: 7", "kitchen", 2499),
    ("P-105", "SwiftMix Mixer Grinder 750W | Rs. 3,499 | 3 jars, overload protection, 2-year warranty | in stock: 23", "kitchen", 3499),
    ("P-107", "AirChef Air Fryer 4L | Rs. 5,999 | oil-free frying, digital timer, 1-year warranty | in stock: 12", "kitchen", 5999),
    ("P-112", "BrightHome Smart Bulb WiFi | Rs. 799 | app control, 16M colors, voice assistant support | in stock: 48", "electrical", 799),
    ("P-115", "EarBuds Bass+ TWS | Rs. 1,499 | 24h playback, IPX4, ENC mic | in stock: 39", "gadgets", 1499),
    ("P-116", "PixelSnap Barcode Scanner X-500 | Rs. 4,999 | wireless, 18h battery, 1-year warranty | in stock: 6", "business", 4999),
    ("P-118", "CleanSweep Robot Vacuum | Rs. 12,999 | app control, auto-dock, HEPA filter | in stock: 3", "home", 12999),
    ("P-119", "FreshBrew Coffee Maker 600ml | Rs. 2,799 | drip brew, keep-warm plate, 1-year warranty | in stock: 14", "kitchen", 2799),
    ("D-201", "POLICY delivery | Free delivery above Rs. 999, otherwise Rs. 49. Standard 2-4 days in Pune, 4-7 days rest of Maharashtra.", "policy", -1),
    ("D-202", "POLICY returns | Returns within 7 days with receipt for full refund. Without receipt: store credit only. Electronics must be unopened.", "policy", -1),
    ("D-203", "POLICY warranty | Warranty claims need invoice + product serial number. Service center: SmartMart Pimple Saudagar, 9 AM-9 PM.", "policy", -1),
    ("D-204", "POLICY offers | Current offer: 10% off all kitchen appliances till Sunday. Not combinable with other coupons.", "policy", -1),
]
IDS  = [r[0] for r in KB]
DOCS = [f"{r[0]} | {r[1]}" for r in KB]

# --- Chroma + BM25 + hybrid (Day 3) ---
chroma = chromadb.Client()
try: chroma.delete_collection("smartmart")
except Exception: pass
col = chroma.create_collection("smartmart", metadata={"hnsw:space": "cosine"})
col.add(ids=IDS, documents=DOCS, embeddings=embed_batch(DOCS).tolist(),
        metadatas=[{"category": r[2], "price": r[3]} for r in KB])

bm25 = BM25Okapi([d.lower().replace("|", " ").replace(",", " ").split() for d in DOCS])

def retrieve_hybrid(query, k=3, pool=8, rrf_k=60):
    scores_bm = bm25.get_scores(query.lower().replace("|", " ").replace(",", " ").split())
    bm_docs = [DOCS[i] for i in sorted(range(len(DOCS)), key=lambda i: scores_bm[i], reverse=True)[:pool]]
    res = col.query(query_embeddings=[embed_batch([query])[0].tolist()], n_results=pool)
    vec_docs = res["documents"][0]
    fused = {}
    for ranking in (bm_docs, vec_docs):
        for rank, doc in enumerate(ranking, 1):
            fused[doc] = fused.get(doc, 0.0) + 1.0 / (rrf_k + rank)
    return [d for d, _ in sorted(fused.items(), key=lambda x: x[1], reverse=True)[:k]]

RAG_TEMPLATE = """You are SmartBot, the customer assistant of SmartMart retail store, Pune.
Answer ONLY from the context below. If the answer is not in the context,
say "I don't have that information — let me connect you with our team."
Cite the product/policy ID you used. Max 3 sentences.

CONTEXT:
{context}

CUSTOMER QUESTION: {question}"""

def rag_answer(question, k=3):
    chunks = retrieve_hybrid(question, k=k)
    answer = ask(RAG_TEMPLATE.format(context="\n".join(chunks), question=question), temperature=0.3)
    return answer, chunks

a, c = rag_answer("cheapest kettle?")
print("✅ v0.3 rebuilt. Smoke test:", a[:90])

---
## Experiment 2 — Faithfulness: catch a hallucination red-handed 🕵️

The RAGAS recipe: **split the answer into atomic claims → verify each against the context → score = supported / total.**

We test it on two answers: a real one from our bot, and one where we **plant a lie**.

In [ ]:
FAITHFULNESS_PROMPT = """You are a strict fact-checking judge.

STEP 1 — Split the ANSWER into its atomic factual claims (short standalone statements).
STEP 2 — For each claim, decide: is it directly supported by the CONTEXT? Answer yes or no.
Ignore politeness, offers of help, and hedges — judge only factual claims.

Return ONLY valid JSON (no markdown fences):
{{"claims": [{{"claim": "...", "supported": true}}, ...]}}

CONTEXT:
{context}

ANSWER:
{answer}"""

def faithfulness(answer, chunks):
    raw = ask(FAITHFULNESS_PROMPT.format(context="\n".join(chunks), answer=answer))
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    data = json.loads(raw)
    claims = data["claims"]
    if not claims:
        return 1.0, []
    score = sum(c["supported"] for c in claims) / len(claims)
    return score, claims

# --- Case A: the bot's real answer ---
question = "What's the warranty on the Prima 1.5L kettle and how do I claim it?"
answer, chunks = rag_answer(question)
score, claims = faithfulness(answer, chunks)
print("🛒 Bot answer:", answer)
print(f"\nFaithfulness: {score:.2f}")
for c in claims:
    print("  ", "✅" if c["supported"] else "❌", c["claim"])

In [ ]:
# --- Case B: the PLANTED hallucination ---
fake_answer = ("The Prima 1.5L kettle comes with a 3-year warranty [P-101]. "
               "Claims are processed online at smartmart.com/warranty within 24 hours, "
               "and you also get a free replacement filter every year.")

score_fake, claims_fake = faithfulness(fake_answer, chunks)
print("🧪 Planted answer:", fake_answer)
print(f"\nFaithfulness: {score_fake:.2f}")
for c in claims_fake:
    print("  ", "✅" if c["supported"] else "❌", c["claim"])

**The judge should flag all three lies** (3-year warranty ≠ 1-year; no online portal in context; no free filter). Notice *how* it caught them: claim-by-claim binary checks — not a vague 1–10 vibe. Binary beats vibes (slide 10).

**✏️ Your turn:** edit `fake_answer` to make ONE lie subtler ("1-year warranty, extendable to 3") and see if the judge still catches it. Welcome to adversarial evaluation.

---
## Experiment 3 — Answer relevancy: the reverse-question trick 🔄

Is the answer *about the question asked*? The RAGAS trick reuses Day 2's embeddings:
**generate 3 questions this answer would answer → embed → cosine vs the real question → mean.**

In [ ]:
REVERSE_Q_PROMPT = """Given this answer from a store assistant, write exactly 3 short questions
that this answer directly and completely answers.
Return ONLY valid JSON (no markdown fences): {{"questions": ["...", "...", "..."]}}

ANSWER:
{answer}"""

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def answer_relevancy(question, answer):
    raw = ask(REVERSE_Q_PROMPT.format(answer=answer))
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    gen_qs = json.loads(raw)["questions"]
    vecs = embed_batch([question] + gen_qs)
    sims = [cosine(vecs[0], v) for v in vecs[1:]]
    return float(np.mean(sims)), gen_qs

# --- Case A: on-topic answer ---
q = "Can I return my kettle without a receipt?"
ans, _ = rag_answer(q)
rel, gen_qs = answer_relevancy(q, ans)
print(f"Q: {q}\n🛒 {ans}\nRelevancy: {rel:.2f}")
for g in gen_qs: print("   reverse-q:", g)

In [ ]:
# --- Case B: FAITHFUL but IRRELEVANT (the sneaky failure) ---
irrelevant_answer = "We're open every day from 9 AM to 9 PM at Pimple Saudagar, Pune [D-203]."
rel_bad, gen_qs_bad = answer_relevancy(q, irrelevant_answer)
print(f"Q: {q}\n🧪 {irrelevant_answer}\nRelevancy: {rel_bad:.2f}")
for g in gen_qs_bad: print("   reverse-q:", g)
print(f"\n→ 100% faithful (store hours ARE true), but relevancy {rel_bad:.2f} vs {rel:.2f}. Faithful is not the same as helpful.")

**Read the reverse questions:** for the bad answer they're all about *store hours* — nowhere near the return question, so cosine sinks. Two metrics, two different failure modes — that's the whole point of a metric *suite*.

---
## Experiment 4 — Grading the context: precision & recall 📦

Now judge what the **retriever** sent, from the answer's perspective:
- **Context precision** — of the k chunks sent, how many were useful for THIS question?
- **Context recall** — does the context contain every fact the *ideal* answer needs? (requires a ground-truth answer)

In [ ]:
CTX_PRECISION_PROMPT = """Question: {question}

For each numbered chunk below, answer: is this chunk USEFUL for answering the question?
A chunk is useful if it contains information needed for the answer.
Return ONLY valid JSON (no markdown fences): {{"useful": [true, false, ...]}} — one per chunk, in order.

CHUNKS:
{chunks}"""

def context_precision(question, chunks):
    numbered = "\n".join(f"{i+1}. {c}" for i, c in enumerate(chunks))
    raw = ask(CTX_PRECISION_PROMPT.format(question=question, chunks=numbered))
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    flags = json.loads(raw)["useful"]
    return sum(flags) / len(flags), flags

CTX_RECALL_PROMPT = """GROUND TRUTH answer: {ground_truth}

For each factual statement in the ground truth, check: can it be found in (or directly inferred from) the CONTEXT below?
Return ONLY valid JSON (no markdown fences):
{{"facts": [{{"fact": "...", "found": true}}, ...]}}

CONTEXT:
{context}"""

def context_recall(ground_truth, chunks):
    raw = ask(CTX_RECALL_PROMPT.format(ground_truth=ground_truth, context="\n".join(chunks)))
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    facts = json.loads(raw)["facts"]
    if not facts:
        return 1.0, []
    return sum(f["found"] for f in facts) / len(facts), facts

# Demo on a question whose ideal answer needs TWO chunks:
q = "What's the price of the X-500 scanner and what do I need for a warranty claim?"
ground_truth = ("The X-500 barcode scanner costs Rs. 4,999. For a warranty claim you need "
                "the original invoice and the product serial number.")
answer, chunks = rag_answer(q)
cp, flags = context_precision(q, chunks)
cr, facts = context_recall(ground_truth, chunks)

print("Retrieved:")
for c, f in zip(chunks, flags):
    print("  ", "✅" if f else "🗑️", c[:70], "...")
print(f"\nContext precision: {cp:.2f}   (useful fraction of what we sent)")
print(f"Context recall   : {cr:.2f}   (needed facts that actually arrived)")
for f in facts:
    print("  ", "✅" if f["found"] else "❌", f["fact"])

**The two context dials in action:** precision counts the 🗑️ junk you paid tokens for; recall checks whether the ideal answer was even *possible* from what arrived. Low recall = no prompt engineering can save you — fix retrieval (triage table!).

**✏️ Your turn:** re-run with `k=1` in `rag_answer`. Precision should rise (less junk), recall should fall (the second fact never arrives). The P/R tension — now on the generation side.

---
## Experiment 5 — The full report card: EvalHarness 📋

Everything wired together over a small eval set with ground truths. **This produces the exact table tomorrow's project requires.**

In [ ]:
EVAL_SET = [
    {"q": "cheapest kettle you have?",
     "truth": "The Zenith Steel Kettle 1.0L is the cheapest at Rs. 899."},
    {"q": "sasta kettle dikhao",
     "truth": "The budget option is the Zenith Steel Kettle 1.0L at Rs. 899."},
    {"q": "can I return my kettle without a receipt?",
     "truth": "Without a receipt you get store credit only; with a receipt, full refund within 7 days."},
    {"q": "X-500 scanner price and warranty claim process?",
     "truth": "The X-500 costs Rs. 4,999. Warranty claims need the original invoice and product serial number."},
    {"q": "is delivery free for a Rs. 500 order in Pune and how long does it take?",
     "truth": "Below Rs. 999 delivery costs Rs. 49. Standard delivery in Pune takes 2-4 days."},
    {"q": "do you sell washing machines?",
     "truth": "SmartMart does not carry washing machines; the assistant should say it doesn't have that information."},
]

def eval_one(item, k=3):
    answer, chunks = rag_answer(item["q"], k=k)
    try: f, _ = faithfulness(answer, chunks)
    except Exception: f = float("nan")
    try: r, _ = answer_relevancy(item["q"], answer)
    except Exception: r = float("nan")
    try: cp, _ = context_precision(item["q"], chunks)
    except Exception: cp = float("nan")
    try: cr, _ = context_recall(item["truth"], chunks)
    except Exception: cr = float("nan")
    return {"question": item["q"], "answer": answer,
            "faithfulness": f, "relevancy": r, "ctx_precision": cp, "ctx_recall": cr}

print("Running the harness (4 judge calls per question — ~30-60s total)...\n")
results = [eval_one(item) for item in EVAL_SET]

print(f"{'question':<44} {'faith':>6} {'relev':>6} {'ctx_P':>6} {'ctx_R':>6}")
print("-" * 72)
for r in results:
    print(f"{r['question'][:42]:<44} {r['faithfulness']:>6.2f} {r['relevancy']:>6.2f} "
          f"{r['ctx_precision']:>6.2f} {r['ctx_recall']:>6.2f}")
print("-" * 72)
means = {m: float(np.nanmean([r[m] for r in results]))
         for m in ["faithfulness", "relevancy", "ctx_precision", "ctx_recall"]}
print(f"{'MEAN':<44} {means['faithfulness']:>6.2f} {means['relevancy']:>6.2f} "
      f"{means['ctx_precision']:>6.2f} {means['ctx_recall']:>6.2f}")

### Reading your report card (the triage table, live)

| Low dial | Fix first |
|---|---|
| `ctx_recall` | chunking (D2) · hybrid/k (D3) · missing documents |
| `ctx_precision` | smaller k · reranking · chunk boundaries |
| `faithfulness` | grounding prompt (D1) · lower temperature · cite rule |
| `relevancy` | prompt TASK part (W1D2) · query rewriting (D3) |

**✏️ Your turn — a real diagnosis:** find your lowest per-question score. Which dial is it? Apply ONE fix from the table, re-run `eval_one` for that question, and note before/after. That before/after pair is exactly what tomorrow's rubric calls "one diagnosed & fixed weakness".

---
## 🏆 Homework (20 min — this IS tomorrow's project prep)

1. **Grow the gold set to 20+** — add at least 7 more eval items *with ground truths*: 2 Hinglish, 2 multi-fact (need 2 chunks), 2 out-of-catalog traps, 1 typo-ridden. Ugly queries are the point.
2. **Judge the judge** — pick 5 judge verdicts (claims or usefulness flags) and verify them yourself. Report: how many do YOU agree with? If <4/5, tighten the rubric prompt and re-run.
3. **The retrieval side** — bring yesterday's retrieval table (P@k/R@k/F1/MRR for BM25/vector/hybrid). Tomorrow's report needs both layers.

### What's next — assembly day 🏗️
**Day 5 — Week 2 Mini-Project (graded):** RAG SmartBot v1.0 — ChromaDB + context-aware chunks + hybrid retrieval + **the full two-layer eval report** you can now generate. Same format as Week 1: workbook, acceptance criteria, rubric, demo.

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*